In [8]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# PHASE 1: THE GATEKEEPER (Processing the Static Data from Excel)
# -------------------------------------------------------------------

# Define the path to your master Excel file
excel_file_path = '../Data/datastram data.xlsx'

# 1. Load the Static Data directly from their specific sheets
# We use header=1 because row 0 is Datastream metadata
alive_static = pd.read_excel(excel_file_path, sheet_name='ALIVE STATIC', header=1)
dead_static = pd.read_excel(excel_file_path, sheet_name='DEAD STATIC', header=1)

# 2. Combine into a single master static dataframe
master_static = pd.concat([alive_static, dead_static], ignore_index=True)

# 3. Apply the Børsprosjektet strict filters
# Keep only Oslo Bors and Equities (EQ)
master_static_filtered = master_static[
    (master_static['BOURSE NAME'] == 'Oslo Bors') & 
    (master_static['STOCK TYPE'] == 'EQ')
].copy()

# 4. Drop any rows that are missing an ISIN
master_static_filtered = master_static_filtered.dropna(subset=['ISIN CODE'])

# 5. Create the "Gatekeeper" Dictionary
# This maps the Datastream 'NAME' to its official 'ISIN CODE'
valid_isin_map = dict(zip(master_static_filtered['NAME'], master_static_filtered['ISIN CODE']))

# Print the results so we can verify the filter worked
print(f"Total raw stocks (Alive + Dead): {len(master_static)}")
print(f"Strictly Oslo Bors Equities (after filter): {len(master_static_filtered)}")
print(f"Gatekeeper dictionary built with {len(valid_isin_map)} mapped ISINs.")

print("\n--- Sample of our Gatekeeper Map ---")
# Show the first 5 mappings to verify
print(list(valid_isin_map.items())[:5])

Total raw stocks (Alive + Dead): 1163
Strictly Oslo Bors Equities (after filter): 911
Gatekeeper dictionary built with 910 mapped ISINs.

--- Sample of our Gatekeeper Map ---
[('AASEN SPAREBANK', 'NO0010672181'), ('ABG SUNDAL COLLIER HOLDING', 'NO0003021909'), ('ABL GROUP', 'NO0010715394'), ('ACE DIGITAL', 'NO0013531616'), ('ADS MARITIME HOLDING', 'CY0108052115')]


In [9]:
# -------------------------------------------------------------------
# PHASE 2 & 3: RESHAPING THE TIME SERIES AND APPLYING THE GATEKEEPER
# -------------------------------------------------------------------

# 1. Load the Series Data from Excel
# We use header=None because the first 5 rows are Datastream metadata 
# (Start, End, Freq, Name, Currency)
alive_series_raw = pd.read_excel(excel_file_path, sheet_name='ALIVE SERIES', header=None)
dead_series_raw = pd.read_excel(excel_file_path, sheet_name='DEAD SERIES', header=None)

# 2. Define a function to melt Datastream's wide matrix into a long panel
def melt_datastream_series(df_raw, valid_map):
    # Extract headers (Row 3 contains the exact base company names)
    headers = df_raw.iloc[3, 1:].values
    
    # Extract dates (Row 5 onwards, Column 0)
    dates = df_raw.iloc[5:, 0].values
    
    # Extract the actual data matrix (Row 5 onwards, Column 1 onwards)
    data_block = df_raw.iloc[5:, 1:].values
    
    records = []
    
    # Iterate through the columns in chunks of 4 (P, RI, NOSH, VO)
    for i in range(0, len(headers), 4):
        # Ensure we don't go out of bounds
        if i + 3 >= len(headers):
            break
            
        # Grab the company name from the first column of the chunk
        company_name = str(headers[i]).strip()
        
        # APPLY THE GATEKEEPER: 
        # If this company is NOT in our Oslo Bors / Equity dictionary, skip it entirely!
        if company_name not in valid_map:
            continue
            
        # If it passed the gatekeeper, get the official ISIN
        isin_code = valid_map[company_name]
        
        # Loop through the dates and extract the time series for this specific stock
        for row_idx, date in enumerate(dates):
            price = data_block[row_idx, i]
            ri = data_block[row_idx, i+1]
            nosh = data_block[row_idx, i+2]
            vo = data_block[row_idx, i+3]
            
            # Only keep rows where the stock actually has a price
            # (Ignores months before an IPO or after a delisting)
            if pd.notna(price) and str(price).strip() != '':
                records.append({
                    'TradeDate': date,
                    'ISIN': isin_code,
                    'Generic': price,
                    'RI': ri,
                    'SharesIssued': nosh,
                    'OffShareTurnover': vo
                })
                
    return pd.DataFrame(records)

# 3. Melt both datasets
print("Melting ALIVE series... (Applying Gatekeeper)")
alive_long = melt_datastream_series(alive_series_raw, valid_isin_map)

print("Melting DEAD series... (Applying Gatekeeper)")
dead_long = melt_datastream_series(dead_series_raw, valid_isin_map)

# 4. Combine them into one master panel
master_series = pd.concat([alive_long, dead_long], ignore_index=True)

print(f"\nSuccessfully melted and purged! Master Series has {len(master_series)} valid rows.")

# Display the first few rows to verify the transformation
display(master_series.head())

Melting ALIVE series... (Applying Gatekeeper)
Melting DEAD series... (Applying Gatekeeper)

Successfully melted and purged! Master Series has 59157 valid rows.


,TradeDate,ISIN,Generic,RI,SharesIssued,OffShareTurnover
0,2020-03-25,NO0010672181,99.0,124.55,1014,1.2
1,2020-04-25,NO0010672181,105.3,132.47,1014,1.1
2,2020-05-25,NO0010672181,102.6,129.08,1014,1.3
3,2020-06-25,NO0010672181,103.5,130.21,1014,0.2
4,2020-07-25,NO0010672181,102.6,129.08,1014,1.3


In [10]:
# -------------------------------------------------------------------
# PHASE 4: SCALING AND FORMATTING TO MATCH BØRSPROSJEKTET
# -------------------------------------------------------------------
print("Starting Phase 4: Formatting and Scaling...")

# 1. Force columns to be numeric 
# (Datastream occasionally leaves blank spaces or string artifacts, so errors='coerce' turns them into clean NaNs)
cols_to_numeric = ['Generic', 'RI', 'SharesIssued', 'OffShareTurnover']
for col in cols_to_numeric:
    master_series[col] = pd.to_numeric(master_series[col], errors='coerce')

# 2. Scale Volume and Shares
# Datastream provides these in thousands. Multiplying by 1,000 matches your legacy data.
master_series['SharesIssued'] = master_series['SharesIssued'] * 1000
master_series['OffShareTurnover'] = master_series['OffShareTurnover'] * 1000

# 3. Format the Dates
# Convert to datetime, then force them to the last day of the month
master_series['TradeDate'] = pd.to_datetime(master_series['TradeDate'])
master_series['TradeDate'] = master_series['TradeDate'] + pd.offsets.MonthEnd(0)

# 4. Add the SecurityType column
# Since our Gatekeeper strictly filtered for 'EQ', we can safely hardcode this to match your legacy data
master_series['SecurityType'] = 'Ordinary Shares'


# -------------------------------------------------------------------
# PHASE 5: CALCULATING THE RETURNS
# -------------------------------------------------------------------
print("Starting Phase 5: Calculating Returns...")

# 1. Sort the data to ensure percentage changes are calculated chronologically per stock
master_series = master_series.sort_values(by=['ISIN', 'TradeDate'])

# 2. Calculate Unadjusted Return (Month-over-Month change in raw Price)
master_series['ReturnGeneric'] = master_series.groupby('ISIN')['Generic'].pct_change()

# 3. Calculate Adjusted Total Return (Month-over-Month change in Return Index)
master_series['ReturnAdjGeneric'] = master_series.groupby('ISIN')['RI'].pct_change()

# 4. Keep only the necessary columns for the final merge
columns_to_keep = [
    'TradeDate', 'ISIN', 'SecurityType', 'Generic', 
    'SharesIssued', 'OffShareTurnover', 'ReturnGeneric', 'ReturnAdjGeneric'
]
ds_final = master_series[columns_to_keep].copy()

# 5. Clean up "Empty" time periods
# Datastream fills all 5 years even if a stock only lived for 1 year. 
# We drop rows where both the Price and the Return are NaN (meaning the stock wasn't trading yet, or is already dead)
ds_final = ds_final.dropna(subset=['Generic', 'ReturnAdjGeneric'], how='all')

print(f"Phases 4 & 5 Complete! The clean 2020-2024 dataset has {len(ds_final)} active trading months.")
display(ds_final.head())

Starting Phase 4: Formatting and Scaling...
Starting Phase 5: Calculating Returns...
Phases 4 & 5 Complete! The clean 2020-2024 dataset has 59157 active trading months.


C:\Users\jakub\AppData\Local\Temp\ipykernel_26704\218673585.py:39: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  master_series['ReturnAdjGeneric'] = master_series.groupby('ISIN')['RI'].pct_change()


,TradeDate,ISIN,SecurityType,Generic,SharesIssued,OffShareTurnover,ReturnGeneric,ReturnAdjGeneric
47059,2020-03-31,ANN7425Q1095,Ordinary Shares,75.5,68548000.0,NaN,NaN,NaN
47060,2020-04-30,ANN7425Q1095,Ordinary Shares,75.5,68548000.0,NaN,0.0,0.0
47061,2020-05-31,ANN7425Q1095,Ordinary Shares,75.5,68548000.0,NaN,0.0,0.0
47062,2020-06-30,ANN7425Q1095,Ordinary Shares,75.5,68548000.0,NaN,0.0,0.0
47063,2020-07-31,ANN7425Q1095,Ordinary Shares,75.5,68548000.0,NaN,0.0,0.0


In [11]:
print(f"Rows before Zombie Exorcism: {len(ds_final)}")

# 1. Drop stocks that died BEFORE 2020 (Total volume in our 5-year window is 0)
stock_volumes = ds_final.groupby('ISIN')['OffShareTurnover'].sum()
alive_in_window = stock_volumes[stock_volumes > 0].index
ds_final = ds_final[ds_final['ISIN'].isin(alive_in_window)]

# 2. Drop "trailing zombies" (Stocks that died during 2020-2024, but Datastream padded their corpse)
# We flag rows where Volume is 0 (or NaN) AND the Price didn't move at all (Return is exactly 0.0)
zombie_mask = (ds_final['OffShareTurnover'].fillna(0) == 0) & (ds_final['ReturnGeneric'] == 0.0)
ds_final = ds_final[~zombie_mask]

print(f"Rows after Zombie Exorcism: {len(ds_final)}")

Rows before Zombie Exorcism: 59157
Rows after Zombie Exorcism: 22563


In [12]:
# -------------------------------------------------------------------
# PHASE 6: THE FINAL MERGE (1980 - 2024)
# -------------------------------------------------------------------
print("Starting Phase 6: The Final Merge...")

# 1. Load the Legacy Børsprosjektet Data
# We use latin1 encoding as per your original Data Cleaning script
legacy_df = pd.read_csv('../Data/monthly_equity_combined.csv', encoding='latin1')

# 2. Rename the first column to 'TradeDate' (matching your original script)
legacy_df.rename(columns={legacy_df.columns[0]: 'TradeDate'}, inplace=True)

# 3. Format Legacy Dates to End of Month
legacy_df['TradeDate'] = pd.to_datetime(legacy_df['TradeDate'], errors='coerce') + pd.offsets.MonthEnd(0)

# 4. Standardize Legacy ISINs
legacy_df['ISIN'] = legacy_df['ISIN'].str.upper().str.strip()

# 5. Subset the legacy data to match our clean Datastream columns exactly
columns_to_keep = [
    'TradeDate', 'ISIN', 'SecurityType', 'Generic', 
    'SharesIssued', 'OffShareTurnover', 'ReturnGeneric', 'ReturnAdjGeneric'
]
legacy_clean = legacy_df[columns_to_keep].copy()

# 6. Concatenate the two datasets! (Stacking them)
master_panel = pd.concat([legacy_clean, ds_final], ignore_index=True)

# 7. Sort the combined dataset chronologically per stock
master_panel = master_panel.sort_values(by=['ISIN', 'TradeDate'])

# 8. Handle any overlapping months (e.g., if both datasets have data for March 2020)
# We drop duplicates based on ISIN and Date. By keeping='last', we trust the 
# newly downloaded Datastream data for any overlapping overlap months.
master_panel = master_panel.drop_duplicates(subset=['ISIN', 'TradeDate'], keep='last')

# 9. Final Cleanup: Drop any rows where ISIN or TradeDate is completely missing
master_panel = master_panel.dropna(subset=['ISIN', 'TradeDate'])

# 10. Save the Master Dataset!
output_filename = '../Data/master_equity_panel_1980_2024.csv'
master_panel.to_csv(output_filename, index=False)

print("\n" + "="*50)
print("SUCCESS! MASTER DATASET CREATED.")
print("="*50)
print(f"Legacy Børsprosjektet Rows: {len(legacy_clean)}")
print(f"New Datastream Rows:        {len(ds_final)}")
print(f"Total Master Panel Rows:    {len(master_panel)}")
print(f"Saved successfully to:      {output_filename}")

# Display the final transition period (where 2020 overlaps) to verify it looks seamless
display(master_panel[(master_panel['TradeDate'] >= '2019-12-31') & (master_panel['TradeDate'] <= '2020-06-30')].head(10))

Starting Phase 6: The Final Merge...

SUCCESS! MASTER DATASET CREATED.
Legacy Børsprosjektet Rows: 162132
New Datastream Rows:        22563
Total Master Panel Rows:    176048
Saved successfully to:      ../Data/master_equity_panel_1980_2024.csv


,TradeDate,ISIN,SecurityType,Generic,SharesIssued,OffShareTurnover,ReturnGeneric,ReturnAdjGeneric
152371,2019-12-31,AU0000057408,Ordinary Shares,1.0200,971665288.0,13296874.0,-0.009516,-0.009709
153280,2020-01-31,AU0000057408,Ordinary Shares,0.9699,971665288.0,12435760.0,-0.049118,-0.049020
154226,2020-02-29,AU0000057408,Ordinary Shares,0.8300,971665288.0,11210916.0,-0.144242,-0.144330
162180,2020-03-31,AU0000057408,Ordinary Shares,0.5400,971665000.0,17333900.0,NaN,NaN
162181,2020-04-30,AU0000057408,Ordinary Shares,0.5800,971665000.0,9581300.0,0.074074,0.092490
162182,2020-05-31,AU0000057408,Ordinary Shares,0.9200,971665000.0,12546900.0,0.586207,0.574530
162183,2020-06-30,AU0000057408,Ordinary Shares,0.9800,971665000.0,10075000.0,0.065217,0.063419
152346,2019-12-31,BMG0451H1170,Ordinary Shares,3.1800,148050298.0,16731501.0,0.104167,0.104167
153255,2020-01-31,BMG0451H1170,Ordinary Shares,2.8900,148050298.0,14994243.0,-0.091195,-0.091195
154201,2020-02-29,BMG0451H1170,Ordinary Shares,2.8450,148050298.0,9690918.0,-0.015571,-0.017301
